In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
url = "https://raw.githubusercontent.com/selva86/datasets/master/Cars93_miss.csv"
df = pd.read_csv(url)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['Price'], kde=True)
plt.title('Price Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.regplot(x='Horsepower', y='Price', data=df)
plt.title('Horsepower vs Price')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.regplot(x='Weight', y='Price', data=df)
plt.title('Weight vs Price')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='Type', y='Price', data=df)
plt.title('Price by Car Type')
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, fmt='.1f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
df = df.drop(['Manufacturer', 'Model', 'Make', 'Min.Price', 'Max.Price'], axis=1)
df = df.dropna(subset=['Price'])
df['Cylinders'] = pd.to_numeric(df['Cylinders'], errors='coerce')
df.shape

In [ ]:
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(exclude='number').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum().sum()

In [ ]:
df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [ ]:
print('Intercept (B0):', model.intercept_)
coef = pd.Series(model.coef_, index=X.columns).sort_values()
coef

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return [mae, mse, rmse, r2]

results = pd.DataFrame(
    [evaluate(y_train, y_train_pred), evaluate(y_test, y_test_pred)],
    columns=['MAE', 'MSE', 'RMSE', 'R2'],
    index=['Train', 'Test']
)
results

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(y_train, y_train_pred, color='blue')
plt.plot([y.min(), y.max()], [y.min(), y.max()], color='red')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Training: Actual vs Predicted')
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(y_test, y_test_pred, color='green')
plt.plot([y.min(), y.max()], [y.min(), y.max()], color='red')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Testing: Actual vs Predicted')
plt.show()

In [ ]:
residuals = y_test_pred - y_test

plt.figure(figsize=(6,4))
plt.scatter(y_test_pred, residuals)
plt.axhline(0, color='red')
plt.xlabel('Predicted Price')
plt.ylabel('Residual')
plt.title('Residual Plot (Test)')
plt.show()